In [0]:
# Import required packages
import pandas as pd
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.getOrCreate()

# Read Delta table
df_spark = spark.read.table("user_behaviour_catalog.user_behaviour_schema.ecomm_behaviour")

# Convert to Pandas
df = df_spark.toPandas()

# Ensure proper datetime format
df['event_time'] = pd.to_datetime(df['event_time'])

# Preview
df.head()


Data Preparation

In [0]:
# Drop rows missing critical info
df = df.dropna(subset=['user_id', 'user_session', 'event_type', 'product_id'])

# Optional: filter to reasonable price range
df = df[(df['price'] > 1) & (df['price'] < 100000)]


Data Transformation – Funnel Creation

In [0]:
# Pivot table to summarize session-level events
funnel = df.pivot_table(
    index=['user_id', 'user_session'],
    columns='event_type',
    values='event_time',
    aggfunc='min'
).reset_index()

# Create binary funnel indicators
funnel['viewed'] = funnel['view'].notna().astype(int)
funnel['carted'] = funnel['cart'].notna().astype(int)
funnel['purchased'] = funnel['purchase'].notna().astype(int)

# Add brand, category_code, price
meta = df.drop_duplicates(subset=['user_id', 'user_session'])[
    ['user_id', 'user_session', 'brand', 'category_code', 'price']
]
funnel = pd.merge(funnel, meta, on=['user_id', 'user_session'], how='left')


Category Hierarchy Extraction

In [0]:
# Split category_code into levels
cat_split = funnel['category_code'].str.split('.', expand=True)
cat_split.columns = ['cat_level_1', 'cat_level_2', 'cat_level_3']
funnel = pd.concat([funnel, cat_split], axis=1)

# Optional: fill missing with "unknown"
funnel[['cat_level_1', 'cat_level_2', 'cat_level_3']] = funnel[[
    'cat_level_1', 'cat_level_2', 'cat_level_3']].fillna('unknown')

funnel.head(5)


ML Model – Predict Purchase Likelihood

In [0]:
%python
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# Prepare features
funnel_ml = funnel[['viewed', 'carted', 'price', 'brand', 'cat_level_1', 'purchased']].dropna()

# Encode categorical variables
le_brand = LabelEncoder()
le_cat1 = LabelEncoder()

funnel_ml['brand_enc'] = le_brand.fit_transform(funnel_ml['brand'].astype(str))


# Fill NaNs first
funnel_ml['cat_level_1'] = funnel_ml['cat_level_1'].fillna('unknown')

# Ensure 1D input
funnel_ml['cat1_enc'] = le_cat1.fit_transform(funnel_ml['cat_level_1'].astype(str))


# Features
X = funnel_ml[['viewed', 'carted', 'price', 'brand_enc', 'cat1_enc']]

# Ensure target is a 1D array (Series)
y = funnel_ml['purchased']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

Problem: Severe Class Imbalance
The model predicts class 0 (no purchase) almost all the time.

Recall for class 1 is only 0.06, meaning only 6% of real purchases were correctly predicted.

This happens despite a decent precision (0.76) on those few it did predict as purchases.

Overall accuracy is misleadingly high at 68% because most users don’t purchase.

Use Class Weights to Penalize Misclassifications
Tell LogisticRegression to care more about class 1:

In [0]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200, class_weight='balanced')
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


Class 1 recall improved from 0.06 → 0.46

F1-score for class 1 went from 0.11 → 0.44

The model is now identifying nearly half of actual purchases.

Balanced performance: You're no longer just predicting "no purchase" for everything.

Reasonable precision and recall for both classes.

A usable model for basic conversion prediction.

Lets try a tree based model 

In [0]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print(classification_report(y_test, y_pred_rf))

Overall Accuracy: 58%
Class 1 Recall: 32% (↓ from 46%)
Class 1 Precision: 36% (↓ from 41%)

The tree-based model shifted slightly toward class 0 again, compared to the logistic regression with class_weight='balanced'.

It likely learned some noise or class imbalance effects, especially if your features are limited or weakly correlated with purchases.

Feature engineering is now key.

In [0]:
import pandas as pd
from pyspark.sql import SparkSession
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

# --------------------------
# 1. Load Delta Table & Convert
# --------------------------
spark = SparkSession.builder.getOrCreate()
df_spark = spark.read.table("user_behaviour_catalog.user_behaviour_schema.ecomm_behaviour")
df = df_spark.toPandas()
df['event_time'] = pd.to_datetime(df['event_time'])

# --------------------------
# 2. Clean and Prepare
# --------------------------
df = df.dropna(subset=['user_id', 'user_session', 'event_type', 'product_id'])
df = df[df['price'].between(1, 100000)]

# --------------------------
# 3. Create Funnel Summary
# --------------------------
funnel = df.pivot_table(
    index=['user_id', 'user_session'],
    columns='event_type',
    values='event_time',
    aggfunc='min'
).reset_index()

funnel['viewed'] = funnel['view'].notna().astype(int)
funnel['carted'] = funnel['cart'].notna().astype(int)
funnel['purchased'] = funnel['purchase'].notna().astype(int)

# --------------------------
# 4. Merge Metadata
# --------------------------
meta = df.drop_duplicates(subset=['user_id', 'user_session'])[
    ['user_id', 'user_session', 'brand', 'category_code', 'price']
]
funnel = funnel.merge(meta, on=['user_id', 'user_session'], how='left')

# --------------------------
# 5. Split category_code into levels
# --------------------------
funnel[['cat_level_1', 'cat_level_2', 'cat_level_3']] = funnel['category_code'].fillna('unknown').astype(str).str.split('.', expand=True)

# --------------------------
# 6. Add Session Features
# --------------------------
session_features = df.groupby(['user_id', 'user_session']).agg(
    event_count=('event_type', 'count'),
    unique_events=('event_type', 'nunique'),
    max_price=('price', 'max'),
    min_price=('price', 'min'),
    mean_price=('price', 'mean')
).reset_index()

funnel = funnel.merge(session_features, on=['user_id', 'user_session'], how='left')

# --------------------------
# 7. Encode Categorical Values
# --------------------------
le_brand = LabelEncoder()
le_cat1 = LabelEncoder()

funnel['brand'] = funnel['brand'].fillna('unknown').astype(str)
funnel['cat_level_1'] = funnel['cat_level_1'].fillna('unknown').astype(str)

funnel['brand_enc'] = le_brand.fit_transform(funnel['brand'])
funnel['cat1_enc'] = le_cat1.fit_transform(funnel['cat_level_1'])

# --------------------------
# 8. Prepare Feature Set
# --------------------------
features = [
    'viewed', 'carted', 'price', 'event_count', 'unique_events',
    'mean_price', 'brand_enc', 'cat1_enc'
]

# Drop rows with missing values in selected features
funnel_ml = funnel.dropna(subset=features + ['purchased'])

X = funnel_ml[features]
y = funnel_ml['purchased'].values.ravel()

# --------------------------
# 9. Train/Test Split
# --------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# --------------------------
# 10. Apply SMOTE to balance training set
# --------------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# --------------------------
# 11. Train Random Forest Model
# --------------------------
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_train_res, y_train_res)

# --------------------------
# 12. Evaluate Model
# --------------------------
y_pred = rf_model.predict(X_test)
report = classification_report(y_test, y_pred)

print("=== CLASSIFICATION REPORT ===")
print(report)


In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# --------------------------
# 13. Confusion Matrix
# --------------------------
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1], yticklabels=[0, 1])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# --------------------------
# 14. Feature Importances
# --------------------------
importances = rf_model.feature_importances_
feature_names = X.columns

# Create a sorted dataframe for plotting
feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot
plt.figure(figsize=(8, 5))
sns.barplot(x='Importance', y='Feature', data=feat_imp_df, palette='viridis')
plt.title('Random Forest Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


Accuracy: 99%

F1 (macro): 0.99

F1 (class 1): 0.99

That’s near-perfect performance — and likely means:

Why This Worked
Feature engineering: session behavior added predictive signal

SMOTE: balanced the classes for learning

Tree model: captured non-linear relationships

###Check for Overfitting

In [0]:
# Evaluate on train set
y_train_pred = rf_model.predict(X_train)
print("TRAIN SET REPORT")
print(classification_report(y_train, y_train_pred))


If train accuracy is 100% and test is also very high, it's okay — only if your features are strong and generalizable.

###Validation

Cross-Validation
Use StratifiedKFold to test generalizability across different splits:

In [0]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf_model, X, y, cv=cv, scoring='f1')

print("Cross-validated F1 scores:", scores)
print("Mean F1:", scores.mean())


False Positive vs. False Negative Analysis
Check: Is it worse to miss a buyer or to target a non-buyer?

False Negatives (missed buyers) → Lost revenue

False Positives (wrongly predicted buyers) → Wasted retargeting effort

In [0]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))


In [0]:
import joblib
joblib.dump(rf_model, 'rf_purchase_predictor.pkl')
joblib.dump(le_brand, 'le_brand.pkl')
joblib.dump(le_cat1, 'le_cat1.pkl')


# Load model
rf_model = joblib.load('rf_purchase_predictor.pkl')
le_brand = joblib.load('le_brand.pkl')
le_cat1  = joblib.load('le_cat1.pkl')

# New session data
new_session = {
    'viewed': 1,
    'carted': 1,
    'price': 0.99,
    'event_count': 4,
    'unique_events': 3,
    'mean_price': 2500000000000.0,
    'brand': 'samsung',
    'cat_level_1': 'electronics'
}

# Encode categorical values
new_session['brand_enc'] = le_brand.transform([new_session['brand']])[0]
new_session['cat1_enc'] = le_cat1.transform([new_session['cat_level_1']])[0]

# Create input vector
X_new = pd.DataFrame([[
    new_session['viewed'],
    new_session['carted'],
    new_session['price'],
    new_session['event_count'],
    new_session['unique_events'],
    new_session['mean_price'],
    new_session['brand_enc'],
    new_session['cat1_enc']
]], columns=[
    'viewed', 'carted', 'price', 'event_count', 'unique_events',
    'mean_price', 'brand_enc', 'cat1_enc'
])

# Predict
pred = rf_model.predict(X_new)[0]
proba = rf_model.predict_proba(X_new)[0][1]
print("Purchase Prediction:", pred)
print("Confidence (probability of purchase):", round(proba, 3))


Business Use Cases from Model Predictions
Personalization: Show better offers/products to likely buyers

Retargeting: Email/SMS for those with high predicted probability but no purchase

UI Optimization: Analyze drop-offs by product/category using model insights

In [0]:
# Predict probabilities for all sessions
funnel_ml['purchase_proba'] = rf_model.predict_proba(X)[:, 1]  # Probability of class 1
funnel_ml.head(5)


Filter High-Likelihood Buyers

In [0]:
# Filter where predicted purchase probability is high (e.g., ≥ 0.7)
likely_buyers = funnel_ml[funnel_ml['purchase_proba'] >= 0.7]

# Preview the result
likely_buyers[['user_id', 'user_session', 'brand', 'cat_level_1', 'purchase_proba']].head()

distinct_users = likely_buyers['user_id'].dropna().unique()
print("Number of distinct likely buyers:", len(distinct_users))
print("Sample user IDs:", distinct_users[:10])


Personalize Offer / Product Recommendation
Here’s a simple logic:

For each likely buyer, recommend another top-selling product in the same category.

In [0]:
# From all purchases, find top brand per category_code
top_brands = df[df['event_type'] == 'purchase'] \
    .groupby('category_code')['brand'] \
    .agg(lambda x: x.value_counts().idxmax()) \
    .reset_index()

top_brands.columns = ['category_code', 'recommended_brand']


# Merge with likely buyers
likely_buyers = pd.merge(
    likely_buyers,
    top_brands,
    on='category_code',
    how='left'
)

likely_buyers[['user_id', 'user_session', 'category_code', 'recommended_brand']].head()

